# 02 — WL-BISINDO Training V4 — FIXED PATH
## Hand134 Dual-Hand Temporal Transformer

Input berasal dari Preprocessing V2:

- `X_hands134.npy`
- `hand_observed_mask.npy`
- `hand_valid_mask.npy`
- `pose_anchor_valid.npy`
- `labels.npy`
- `signer_ids.npy`
- `sample_ids.npy`
- `metadata.csv`
- `class_mapping.json`

### Protocol

- signer 4 = **final unseen test**
- signer 4 tidak disentuh untuk epoch selection
- development validation diusahakan **unseen-signer** juga:
  - notebook mencari signer 0–3 yang memiliki semua 32 kelas
  - signer tersebut dipakai sebagai development validation
  - signer lain menjadi development train
- jika tidak ada signer development yang memiliki 32 kelas penuh, otomatis fallback ke stratified per-class validation
- setelah `best_epoch` ditemukan, model diinisialisasi ulang dan dilatih menggunakan **semua signer 0–3**
- baru kemudian signer 4 diuji satu kali

Arsitektur:

`Left Hand Encoder + Right Hand Encoder → Fusion → Temporal Conv → Transformer → 32 classes`

In [ ]:
from pathlib import Path
import os
import json
import random
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings(
    "ignore"
)

SEED = 42

SEQ_LEN = 48
HAND_FEATURES = 67
FEATURE_DIM = 134
NUM_CLASSES = 32

TEST_SIGNER = 4

BATCH_SIZE = 64
NUM_WORKERS = 2

MAX_DEV_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15

HAND_EMBED = 96
D_MODEL = 192
NHEAD = 6
NUM_LAYERS = 3
DIM_FF = 384
DROPOUT = 0.25

LR = 3e-4
WEIGHT_DECAY = 1e-3
LABEL_SMOOTHING = 0.03
GRAD_CLIP = 1.0

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AMP_ENABLED = (
    DEVICE.type
    == "cuda"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "wl_bisindo_hand134_transformer_v4"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Python :",
    os.sys.version.split()[0]
)

print(
    "NumPy  :",
    np.__version__
)

print(
    "Pandas :",
    pd.__version__
)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "Device :",
    DEVICE
)

if DEVICE.type == "cuda":
    print(
        "GPU    :",
        torch.cuda.get_device_name(0)
    )

In [ ]:
def seed_everything(
    seed=42
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    torch.cuda.manual_seed_all(
        seed
    )

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(
    SEED
)

## Preprocessing V2 input path — FIXED

Path dikunci ke:

`/kaggle/input/datasets/loliwibu/preprocessing-output-bisindo`

In [ ]:
LANDMARK_DIR = Path(
    "/kaggle/input/datasets/loliwibu/preprocessing-output-bisindo"
)

print("Using:", LANDMARK_DIR)

if not LANDMARK_DIR.exists():
    raise FileNotFoundError(
        f"Preprocessing directory tidak ditemukan: {LANDMARK_DIR}"
    )

required = [
    "X_hands134.npy",
    "hand_observed_mask.npy",
    "hand_valid_mask.npy",
    "pose_anchor_valid.npy",
    "labels.npy",
    "signer_ids.npy",
    "sample_ids.npy",
    "metadata.csv",
    "class_mapping.json",
]

missing = [
    name
    for name in required
    if not (LANDMARK_DIR / name).exists()
]

if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}"
    )

print("✅ All preprocessing V2 files found")

for name in required:
    p = LANDMARK_DIR / name
    print(
        f" - {name:<26} "
        f"{p.stat().st_size/(1024**2):8.2f} MB"
    )

In [ ]:
X_raw = np.load(
    LANDMARK_DIR
    / "X_hands134.npy",
    mmap_mode="r"
)

OBS_raw = np.load(
    LANDMARK_DIR
    / "hand_observed_mask.npy",
    mmap_mode="r"
)

VALID_raw = np.load(
    LANDMARK_DIR
    / "hand_valid_mask.npy",
    mmap_mode="r"
)

POSE_raw = np.load(
    LANDMARK_DIR
    / "pose_anchor_valid.npy",
    mmap_mode="r"
)

labels = np.load(
    LANDMARK_DIR
    / "labels.npy"
).astype(
    np.int64
)

signers = np.load(
    LANDMARK_DIR
    / "signer_ids.npy"
).astype(
    np.int64
)

sample_ids = np.load(
    LANDMARK_DIR
    / "sample_ids.npy"
).astype(
    np.int64
)

metadata = pd.read_csv(
    LANDMARK_DIR
    / "metadata.csv"
)

with open(
    LANDMARK_DIR
    / "class_mapping.json",
    "r",
    encoding="utf-8",
) as f:
    mapping_json = json.load(
        f
    )

LABELS_ID = {
    int(k): v
    for k, v
    in mapping_json.items()
}

print(
    "X      :",
    X_raw.shape
)

print(
    "OBS    :",
    OBS_raw.shape
)

print(
    "VALID  :",
    VALID_raw.shape
)

print(
    "POSE   :",
    POSE_raw.shape
)

print(
    "Labels :",
    labels.shape
)

print(
    "Signers:",
    sorted(
        np.unique(
            signers
        ).tolist()
    )
)

print(
    "Classes:",
    len(
        np.unique(
            labels
        )
    )
)

assert X_raw.shape == (
    1600,
    48,
    134
)

assert OBS_raw.shape == (
    1600,
    48,
    2
)

assert VALID_raw.shape == (
    1600,
    48,
    2
)

assert labels.shape == (
    1600,
)

assert signers.shape == (
    1600,
)

assert sorted(
    np.unique(
        signers
    ).tolist()
) == [
    0, 1, 2, 3, 4
]

assert sorted(
    np.unique(
        labels
    ).tolist()
) == list(
    range(
        NUM_CLASSES
    )
)

assert np.isfinite(
    X_raw
).all()

print(
    "✅ Input integrity PASSED"
)

## Signer-independent development protocol

Prioritas: development validation juga **unseen signer**, agar `best_epoch` tidak terlalu optimistis.

Notebook akan otomatis mencari signer 0–3 yang punya semua 32 kelas.

In [ ]:
all_idx = np.arange(
    len(labels),
    dtype=np.int64
)

dev_pool_idx = all_idx[
    signers != TEST_SIGNER
]

test_idx = all_idx[
    signers == TEST_SIGNER
]

assert len(
    dev_pool_idx
) == 1280

assert len(
    test_idx
) == 320

# Audit signer x class counts.
count_matrix = np.zeros(
    (
        5,
        NUM_CLASSES
    ),
    dtype=np.int64
)

for signer_id in range(5):
    for label_id in range(
        NUM_CLASSES
    ):
        count_matrix[
            signer_id,
            label_id
        ] = int(
            np.sum(
                (
                    signers
                    == signer_id
                )
                & (
                    labels
                    == label_id
                )
            )
        )

count_df = pd.DataFrame(
    count_matrix,
    index=[
        f"signer_{i}"
        for i in range(5)
    ],
    columns=[
        LABELS_ID[i]
        for i in range(
            NUM_CLASSES
        )
    ],
)

display(
    count_df
)

dev_signer_candidates = []

for signer_id in [
    0, 1, 2, 3
]:
    counts = count_matrix[
        signer_id
    ]

    if np.all(
        counts > 0
    ):
        dev_signer_candidates.append({
            "signer_id": signer_id,
            "min_per_class": int(
                counts.min()
            ),
            "total": int(
                counts.sum()
            ),
        })

print(
    "Full-class development signer candidates:",
    dev_signer_candidates
)

if dev_signer_candidates:
    # Prefer the most balanced complete signer.
    chosen = sorted(
        dev_signer_candidates,
        key=lambda x: (
            x["min_per_class"],
            x["total"]
        ),
        reverse=True
    )[0]

    DEV_VAL_SIGNER = int(
        chosen[
            "signer_id"
        ]
    )

    val_idx = dev_pool_idx[
        signers[
            dev_pool_idx
        ] == DEV_VAL_SIGNER
    ]

    train_idx = dev_pool_idx[
        signers[
            dev_pool_idx
        ] != DEV_VAL_SIGNER
    ]

    SPLIT_MODE = (
        "leave-one-development-signer-out"
    )

else:
    DEV_VAL_SIGNER = None
    SPLIT_MODE = (
        "stratified-per-class-fallback"
    )

    VAL_PER_CLASS = 4
    rng = np.random.default_rng(
        SEED
    )

    val_list = []

    for label_id in range(
        NUM_CLASSES
    ):
        candidates = dev_pool_idx[
            labels[
                dev_pool_idx
            ] == label_id
        ].copy()

        if len(
            candidates
        ) <= VAL_PER_CLASS:
            raise RuntimeError(
                f"Insufficient development samples "
                f"for class {label_id}"
            )

        rng.shuffle(
            candidates
        )

        selected = candidates[
            :VAL_PER_CLASS
        ]

        val_list.extend(
            selected.tolist()
        )

    val_idx = np.asarray(
        sorted(
            val_list
        ),
        dtype=np.int64
    )

    train_idx = np.asarray(
        sorted(
            set(
                dev_pool_idx.tolist()
            )
            - set(
                val_idx.tolist()
            )
        ),
        dtype=np.int64
    )

# Integrity checks.
assert set(
    train_idx.tolist()
).isdisjoint(
    set(
        val_idx.tolist()
    )
)

assert set(
    dev_pool_idx.tolist()
).isdisjoint(
    set(
        test_idx.tolist()
    )
)

assert sorted(
    np.unique(
        labels[
            train_idx
        ]
    ).tolist()
) == list(
    range(
        NUM_CLASSES
    )
)

assert sorted(
    np.unique(
        labels[
            val_idx
        ]
    ).tolist()
) == list(
    range(
        NUM_CLASSES
    )
)

assert sorted(
    np.unique(
        labels[
            test_idx
        ]
    ).tolist()
) == list(
    range(
        NUM_CLASSES
    )
)

print()
print(
    "=" * 72
)

print(
    "DEVELOPMENT / FINAL TEST SPLIT"
)

print(
    "=" * 72
)

print(
    "Mode       :",
    SPLIT_MODE
)

print(
    "Dev val ID :",
    DEV_VAL_SIGNER
)

print(
    "Dev train  :",
    len(
        train_idx
    )
)

print(
    "Dev val    :",
    len(
        val_idx
    )
)

print(
    "Full train :",
    len(
        dev_pool_idx
    )
)

print(
    "Final test :",
    len(
        test_idx
    )
)

for name, idx in [
    (
        "dev_train",
        train_idx
    ),
    (
        "dev_val",
        val_idx
    ),
    (
        "final_test",
        test_idx
    ),
]:
    print(
        f"{name:<10} | "
        f"signers="
        f"{sorted(np.unique(signers[idx]).tolist())} | "
        f"classes="
        f"{len(np.unique(labels[idx]))}"
    )

print(
    "✅ SI development split PASSED"
)

## Feature standardization

Per hand:

- indices `0:63` = local landmark
- `63:66` = wrist global
- `66` = presence

Left block = `0:67`  
Right block = `67:134`

Presence bits tidak distandardisasi.

In [ ]:
LEFT_START = 0
RIGHT_START = 67

LEFT_PRESENCE_IDX = 66
RIGHT_PRESENCE_IDX = 133

PRESENCE_IDXS = [
    LEFT_PRESENCE_IDX,
    RIGHT_PRESENCE_IDX,
]

continuous_mask = np.ones(
    FEATURE_DIM,
    dtype=bool
)

continuous_mask[
    PRESENCE_IDXS
] = False

train_values = np.asarray(
    X_raw[
        train_idx
    ]
)

feature_mean = np.zeros(
    FEATURE_DIM,
    dtype=np.float32
)

feature_std = np.ones(
    FEATURE_DIM,
    dtype=np.float32
)

feature_mean[
    continuous_mask
] = train_values[
    :,
    :,
    continuous_mask
].mean(
    axis=(0, 1)
).astype(
    np.float32
)

feature_std[
    continuous_mask
] = train_values[
    :,
    :,
    continuous_mask
].std(
    axis=(0, 1)
).astype(
    np.float32
)

feature_std = np.where(
    feature_std
    < 1e-5,
    1.0,
    feature_std
).astype(
    np.float32
)

np.save(
    OUTPUT_DIR
    / "feature_mean_dev.npy",
    feature_mean
)

np.save(
    OUTPUT_DIR
    / "feature_std_dev.npy",
    feature_std
)

print(
    "Mean:",
    feature_mean.shape
)

print(
    "Std :",
    feature_std.shape
)

## Dataset + augmentation

In [ ]:
def temporal_resample(
    x,
    factor
):
    T, F = x.shape

    old_t = np.arange(
        T,
        dtype=np.float32
    )

    center = (
        T - 1
    ) / 2.0

    new_t = (
        (
            np.arange(
                T,
                dtype=np.float32
            )
            - center
        )
        / factor
        + center
    )

    new_t = np.clip(
        new_t,
        0,
        T - 1
    )

    out = np.empty_like(
        x
    )

    for f in range(
        F
    ):
        out[
            :,
            f
        ] = np.interp(
            new_t,
            old_t,
            x[
                :,
                f
            ]
        )

    return out


class SignDataset(
    Dataset
):
    def __init__(
        self,
        indices,
        training,
        mean,
        std,
    ):
        self.indices = np.asarray(
            indices,
            dtype=np.int64
        )

        self.training = training
        self.mean = mean
        self.std = std

    def __len__(
        self
    ):
        return len(
            self.indices
        )

    def __getitem__(
        self,
        i
    ):
        rid = int(
            self.indices[i]
        )

        x = np.asarray(
            X_raw[
                rid
            ]
        ).copy()

        left_presence = x[
            :,
            LEFT_PRESENCE_IDX
        ].copy()

        right_presence = x[
            :,
            RIGHT_PRESENCE_IDX
        ].copy()

        if self.training:
            # Speed.
            if np.random.rand() < 0.50:
                speed = np.random.uniform(
                    0.90,
                    1.10
                )

                x = temporal_resample(
                    x,
                    speed
                )

                left_presence = (
                    temporal_resample(
                        left_presence[
                            :,
                            None
                        ],
                        speed
                    )[
                        :,
                        0
                    ]
                    >= 0.5
                ).astype(
                    np.float32
                )

                right_presence = (
                    temporal_resample(
                        right_presence[
                            :,
                            None
                        ],
                        speed
                    )[
                        :,
                        0
                    ]
                    >= 0.5
                ).astype(
                    np.float32
                )

            # Temporal shift.
            if np.random.rand() < 0.50:
                shift = np.random.randint(
                    -3,
                    4
                )

                x = np.roll(
                    x,
                    shift,
                    axis=0
                )

                left_presence = np.roll(
                    left_presence,
                    shift
                )

                right_presence = np.roll(
                    right_presence,
                    shift
                )

            # Coordinate noise.
            if np.random.rand() < 0.60:
                noise = np.random.normal(
                    0.0,
                    0.005,
                    size=x.shape,
                ).astype(
                    np.float32
                )

                noise[
                    :,
                    LEFT_PRESENCE_IDX
                ] = 0

                noise[
                    :,
                    RIGHT_PRESENCE_IDX
                ] = 0

                x = (
                    x
                    + noise
                )

            # Very short frame masking.
            if np.random.rand() < 0.25:
                span = np.random.randint(
                    1,
                    4
                )

                start = np.random.randint(
                    0,
                    SEQ_LEN
                    - span
                    + 1
                )

                x[
                    start:
                    start + span
                ] = 0.0

                left_presence[
                    start:
                    start + span
                ] = 0.0

                right_presence[
                    start:
                    start + span
                ] = 0.0

            # Occasional single-hand dropout for robustness.
            if np.random.rand() < 0.10:
                side = np.random.randint(
                    0,
                    2
                )

                span = np.random.randint(
                    1,
                    4
                )

                start = np.random.randint(
                    0,
                    SEQ_LEN
                    - span
                    + 1
                )

                if side == 0:
                    x[
                        start:
                        start + span,
                        0:67
                    ] = 0.0

                    left_presence[
                        start:
                        start + span
                    ] = 0.0

                else:
                    x[
                        start:
                        start + span,
                        67:134
                    ] = 0.0

                    right_presence[
                        start:
                        start + span
                    ] = 0.0

        # Normalize.
        x = (
            x
            - self.mean
        ) / self.std

        left_absent = (
            left_presence
            < 0.5
        )

        right_absent = (
            right_presence
            < 0.5
        )

        # Zero continuous feature blocks when hand is absent.
        x[
            left_absent,
            0:66
        ] = 0.0

        x[
            right_absent,
            67:133
        ] = 0.0

        x[
            :,
            LEFT_PRESENCE_IDX
        ] = left_presence

        x[
            :,
            RIGHT_PRESENCE_IDX
        ] = right_presence

        y = int(
            labels[
                rid
            ]
        )

        return (
            torch.from_numpy(
                x.astype(
                    np.float32
                )
            ),
            torch.tensor(
                y,
                dtype=torch.long
            ),
        )

In [ ]:
train_ds = SignDataset(
    train_idx,
    training=True,
    mean=feature_mean,
    std=feature_std,
)

val_ds = SignDataset(
    val_idx,
    training=False,
    mean=feature_mean,
    std=feature_std,
)

test_ds = SignDataset(
    test_idx,
    training=False,
    mean=feature_mean,
    std=feature_std,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED,
)

xb, yb = next(
    iter(
        train_loader
    )
)

print(
    "Batch X:",
    xb.shape
)

print(
    "Batch y:",
    yb.shape
)

assert xb.shape[
    1:
] == (
    SEQ_LEN,
    FEATURE_DIM
)

print(
    "✅ DataLoader PASSED"
)

## Model — Dual-Hand Temporal Transformer

Left dan right hand diberi encoder terpisah sebelum difusikan. Ini mencegah model memperlakukan 134 fitur sebagai satu blok mentah.

In [ ]:
class DualHandTemporalTransformer(
    nn.Module
):
    def __init__(
        self,
        num_classes=32,
        seq_len=48,
        hand_embed=96,
        d_model=192,
        nhead=6,
        num_layers=3,
        dim_ff=384,
        dropout=0.25,
    ):
        super().__init__()

        self.left_encoder = nn.Sequential(
            nn.Linear(
                67,
                hand_embed
            ),
            nn.LayerNorm(
                hand_embed
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
        )

        self.right_encoder = nn.Sequential(
            nn.Linear(
                67,
                hand_embed
            ),
            nn.LayerNorm(
                hand_embed
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
        )

        self.fusion = nn.Sequential(
            nn.Linear(
                hand_embed
                * 2,
                d_model
            ),
            nn.LayerNorm(
                d_model
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
        )

        self.temporal_conv = nn.Sequential(
            nn.Conv1d(
                d_model,
                d_model,
                kernel_size=3,
                padding=1,
                groups=d_model,
            ),
            nn.Conv1d(
                d_model,
                d_model,
                kernel_size=1,
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
        )

        self.cls_token = nn.Parameter(
            torch.zeros(
                1,
                1,
                d_model
            )
        )

        self.pos_embed = nn.Parameter(
            torch.zeros(
                1,
                seq_len + 1,
                d_model
            )
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_ff,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=num_layers,
            )
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        self.head = nn.Sequential(
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                d_model,
                num_classes
            ),
        )

        nn.init.trunc_normal_(
            self.cls_token,
            std=0.02
        )

        nn.init.trunc_normal_(
            self.pos_embed,
            std=0.02
        )

    def forward(
        self,
        x
    ):
        left = x[
            :,
            :,
            0:67
        ]

        right = x[
            :,
            :,
            67:134
        ]

        left = (
            self.left_encoder(
                left
            )
        )

        right = (
            self.right_encoder(
                right
            )
        )

        x = torch.cat(
            [
                left,
                right
            ],
            dim=-1
        )

        x = self.fusion(
            x
        )

        conv = self.temporal_conv(
            x.transpose(
                1,
                2
            )
        ).transpose(
            1,
            2
        )

        x = (
            x
            + conv
        )

        cls = self.cls_token.expand(
            x.size(0),
            -1,
            -1
        )

        x = torch.cat(
            [
                cls,
                x
            ],
            dim=1
        )

        x = (
            x
            + self.pos_embed
        )

        x = self.encoder(
            x
        )

        x = self.norm(
            x[
                :,
                0
            ]
        )

        return self.head(
            x
        )


def build_model():
    return DualHandTemporalTransformer(
        num_classes=NUM_CLASSES,
        seq_len=SEQ_LEN,
        hand_embed=HAND_EMBED,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_layers=NUM_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
    )


model = build_model().to(
    DEVICE
)

print(
    "Params:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

## Development training

In [ ]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=LABEL_SMOOTHING
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
    )
)

scaler = (
    torch.cuda.amp
    .GradScaler(
        enabled=AMP_ENABLED
    )
)

DEV_BEST_PATH = (
    OUTPUT_DIR
    / "dev_best.pt"
)


def run_epoch(
    model,
    loader,
    optimizer=None,
    scaler_obj=None,
):
    training = (
        optimizer
        is not None
    )

    if training:
        model.train()

    else:
        model.eval()

    total_loss = 0.0
    yt = []
    yp = []

    for x, y in loader:
        x = x.to(
            DEVICE,
            non_blocking=True
        )

        y = y.to(
            DEVICE,
            non_blocking=True
        )

        if training:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(
            training
        ):
            with torch.cuda.amp.autocast(
                enabled=AMP_ENABLED
            ):
                logits = model(
                    x
                )

                loss = criterion(
                    logits,
                    y
                )

            if training:
                scaler_obj.scale(
                    loss
                ).backward()

                scaler_obj.unscale_(
                    optimizer
                )

                nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRAD_CLIP
                )

                scaler_obj.step(
                    optimizer
                )

                scaler_obj.update()

        total_loss += (
            loss.item()
            * x.size(0)
        )

        pred = logits.argmax(
            dim=1
        )

        yt.extend(
            y.detach()
            .cpu()
            .tolist()
        )

        yp.extend(
            pred.detach()
            .cpu()
            .tolist()
        )

    return {
        "loss": (
            total_loss
            / len(
                loader.dataset
            )
        ),
        "accuracy": accuracy_score(
            yt,
            yp
        ),
        "macro_f1": f1_score(
            yt,
            yp,
            average="macro",
            zero_division=0,
        ),
    }

In [ ]:
history = []

best_val_f1 = -1.0
best_epoch = 1
wait = 0

for epoch in range(
    1,
    MAX_DEV_EPOCHS + 1
):
    train_metrics = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scaler_obj=scaler,
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        optimizer=None,
        scaler_obj=None,
    )

    scheduler.step(
        val_metrics[
            "macro_f1"
        ]
    )

    lr_now = (
        optimizer
        .param_groups[0][
            "lr"
        ]
    )

    row = {
        "epoch": epoch,
        "lr": lr_now,
        "train_loss": train_metrics[
            "loss"
        ],
        "train_accuracy": train_metrics[
            "accuracy"
        ],
        "train_macro_f1": train_metrics[
            "macro_f1"
        ],
        "val_loss": val_metrics[
            "loss"
        ],
        "val_accuracy": val_metrics[
            "accuracy"
        ],
        "val_macro_f1": val_metrics[
            "macro_f1"
        ],
    }

    history.append(
        row
    )

    print(
        f"Epoch {epoch:03d} | "
        f"train acc="
        f"{train_metrics['accuracy']:.4f} "
        f"F1="
        f"{train_metrics['macro_f1']:.4f} | "
        f"val acc="
        f"{val_metrics['accuracy']:.4f} "
        f"F1="
        f"{val_metrics['macro_f1']:.4f} | "
        f"lr={lr_now:.2e}"
    )

    if (
        val_metrics[
            "macro_f1"
        ]
        > best_val_f1
    ):
        best_val_f1 = float(
            val_metrics[
                "macro_f1"
            ]
        )

        best_epoch = epoch
        wait = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": (
                    model.state_dict()
                ),
                "best_val_macro_f1": (
                    best_val_f1
                ),
                "split_mode": (
                    SPLIT_MODE
                ),
                "dev_val_signer": (
                    DEV_VAL_SIGNER
                ),
            },
            DEV_BEST_PATH,
        )

        print(
            "  ✅ best development model"
        )

    else:
        wait += 1

        print(
            f"  patience "
            f"{wait}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

    if (
        wait
        >= EARLY_STOPPING_PATIENCE
    ):
        print(
            "⏹️ Development early stopping"
        )

        break

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    OUTPUT_DIR
    / "development_history.csv",
    index=False
)

print()
print(
    "BEST EPOCH :",
    best_epoch
)

print(
    "BEST VAL F1:",
    round(
        best_val_f1,
        4
    )
)

In [ ]:
plt.figure(
    figsize=(
        9,
        5
    )
)

plt.plot(
    history_df.epoch,
    history_df.train_macro_f1,
    marker="o",
    label="Train F1",
)

plt.plot(
    history_df.epoch,
    history_df.val_macro_f1,
    marker="o",
    label="Validation F1",
)

plt.axvline(
    best_epoch,
    linestyle="--",
    label=(
        f"Best epoch "
        f"{best_epoch}"
    ),
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Macro-F1"
)

plt.ylim(
    0,
    1
)

plt.legend()
plt.grid(
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "development_f1.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()

## Final retraining — signer 0–3

Setelah best epoch ditemukan dari development protocol, mean/std dihitung ulang dari seluruh signer 0–3. Model diinisialisasi ulang dari nol lalu dilatih tepat `best_epoch`.

In [ ]:
dev_values = np.asarray(
    X_raw[
        dev_pool_idx
    ]
)

final_mean = np.zeros(
    FEATURE_DIM,
    dtype=np.float32
)

final_std = np.ones(
    FEATURE_DIM,
    dtype=np.float32
)

final_mean[
    continuous_mask
] = dev_values[
    :,
    :,
    continuous_mask
].mean(
    axis=(0, 1)
).astype(
    np.float32
)

final_std[
    continuous_mask
] = dev_values[
    :,
    :,
    continuous_mask
].std(
    axis=(0, 1)
).astype(
    np.float32
)

final_std = np.where(
    final_std
    < 1e-5,
    1.0,
    final_std
).astype(
    np.float32
)

np.save(
    OUTPUT_DIR
    / "feature_mean.npy",
    final_mean
)

np.save(
    OUTPUT_DIR
    / "feature_std.npy",
    final_std
)

full_train_ds = SignDataset(
    dev_pool_idx,
    training=True,
    mean=final_mean,
    std=final_std,
)

final_test_ds = SignDataset(
    test_idx,
    training=False,
    mean=final_mean,
    std=final_std,
)

full_train_loader = DataLoader(
    full_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED,
)

final_test_loader = DataLoader(
    final_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED,
)

seed_everything(
    SEED
)

final_model = (
    build_model()
    .to(
        DEVICE
    )
)

final_optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

final_scheduler = (
    torch.optim.lr_scheduler
    .CosineAnnealingLR(
        final_optimizer,
        T_max=max(
            best_epoch,
            1
        ),
        eta_min=1e-6,
    )
)

final_scaler = (
    torch.cuda.amp
    .GradScaler(
        enabled=AMP_ENABLED
    )
)

final_history = []

for epoch in range(
    1,
    best_epoch + 1
):
    metrics = run_epoch(
        final_model,
        full_train_loader,
        optimizer=final_optimizer,
        scaler_obj=final_scaler,
    )

    final_scheduler.step()

    final_history.append({
        "epoch": epoch,
        **metrics,
    })

    print(
        f"Final "
        f"{epoch:03d}/"
        f"{best_epoch} | "
        f"loss="
        f"{metrics['loss']:.4f} "
        f"acc="
        f"{metrics['accuracy']:.4f} "
        f"F1="
        f"{metrics['macro_f1']:.4f}"
    )

pd.DataFrame(
    final_history
).to_csv(
    OUTPUT_DIR
    / "final_training_history.csv",
    index=False
)

FINAL_MODEL_PATH = (
    OUTPUT_DIR
    / "hand134_transformer_final.pt"
)

torch.save(
    {
        "model_state_dict": (
            final_model
            .state_dict()
        ),
        "best_epoch": int(
            best_epoch
        ),
        "labels": LABELS_ID,
        "seq_len": SEQ_LEN,
        "feature_dim": FEATURE_DIM,
        "test_signer": TEST_SIGNER,
        "split_mode": SPLIT_MODE,
        "dev_val_signer": (
            DEV_VAL_SIGNER
        ),
    },
    FINAL_MODEL_PATH,
)

print(
    "✅ Final checkpoint:",
    FINAL_MODEL_PATH
)

## Final unseen-signer test

In [ ]:
@torch.no_grad()
def evaluate(
    model,
    loader
):
    model.eval()

    yt = []
    yp = []
    confs = []

    for x, y in loader:
        x = x.to(
            DEVICE,
            non_blocking=True
        )

        with torch.cuda.amp.autocast(
            enabled=AMP_ENABLED
        ):
            logits = model(
                x
            )

        probs = torch.softmax(
            logits,
            dim=1
        )

        conf, pred = probs.max(
            dim=1
        )

        yt.extend(
            y.tolist()
        )

        yp.extend(
            pred.cpu()
            .tolist()
        )

        confs.extend(
            conf.cpu()
            .tolist()
        )

    return (
        np.asarray(
            yt
        ),
        np.asarray(
            yp
        ),
        np.asarray(
            confs
        ),
    )


y_true, y_pred, y_conf = evaluate(
    final_model,
    final_test_loader
)

test_acc = accuracy_score(
    y_true,
    y_pred
)

test_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0,
)

print(
    "=" * 72
)

print(
    "FINAL SIGNER-INDEPENDENT TEST"
)

print(
    "=" * 72
)

print(
    "Test signer   :",
    TEST_SIGNER
)

print(
    f"TEST ACCURACY : "
    f"{test_acc:.4f}"
)

print(
    f"TEST MACRO-F1 : "
    f"{test_f1:.4f}"
)

In [ ]:
report = classification_report(
    y_true,
    y_pred,
    labels=list(
        range(
            NUM_CLASSES
        )
    ),
    target_names=[
        LABELS_ID[i]
        for i in range(
            NUM_CLASSES
        )
    ],
    zero_division=0,
    output_dict=True,
)

report_df = pd.DataFrame(
    report
).transpose()

report_df.to_csv(
    OUTPUT_DIR
    / "classification_report.csv"
)

display(
    report_df
)

class_rows = []

for class_id in range(
    NUM_CLASSES
):
    class_name = (
        LABELS_ID[
            class_id
        ]
    )

    row = report_df.loc[
        class_name
    ]

    class_rows.append({
        "class_id": class_id,
        "class_name": class_name,
        "precision": float(
            row[
                "precision"
            ]
        ),
        "recall": float(
            row[
                "recall"
            ]
        ),
        "f1": float(
            row[
                "f1-score"
            ]
        ),
        "support": int(
            row[
                "support"
            ]
        ),
    })

class_metrics = pd.DataFrame(
    class_rows
)

class_metrics.to_csv(
    OUTPUT_DIR
    / "per_class_metrics.csv",
    index=False
)

print(
    "Worst classes:"
)

display(
    class_metrics
    .sort_values(
        "f1"
    )
    .head(10)
)

zero_f1 = class_metrics[
    class_metrics[
        "f1"
    ] <= 1e-12
]

print(
    "Zero-F1 classes:",
    zero_f1[
        "class_name"
    ].tolist()
)

prediction_df = pd.DataFrame({
    "row_id": test_idx,
    "true_id": y_true,
    "pred_id": y_pred,
    "confidence": y_conf,
})

prediction_df[
    "true_label"
] = [
    LABELS_ID[
        int(i)
    ]
    for i in y_true
]

prediction_df[
    "pred_label"
] = [
    LABELS_ID[
        int(i)
    ]
    for i in y_pred
]

prediction_df.to_csv(
    OUTPUT_DIR
    / "test_predictions.csv",
    index=False
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(
        range(
            NUM_CLASSES
        )
    ),
)

np.save(
    OUTPUT_DIR
    / "confusion_matrix.npy",
    cm
)

plt.figure(
    figsize=(
        18,
        16
    )
)

plt.imshow(
    cm,
    interpolation="nearest"
)

plt.colorbar()

plt.title(
    "WL-BISINDO Hand134 Transformer V4"
)

ticks = np.arange(
    NUM_CLASSES
)

plt.xticks(
    ticks,
    [
        LABELS_ID[i]
        for i in range(
            NUM_CLASSES
        )
    ],
    rotation=90,
)

plt.yticks(
    ticks,
    [
        LABELS_ID[i]
        for i in range(
            NUM_CLASSES
        )
    ],
)

plt.xlabel(
    "Predicted"
)

plt.ylabel(
    "True"
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "confusion_matrix.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()

## TorchScript export — PyTorch 2.10 safe

In [ ]:
export_model = (
    final_model
    .eval()
    .cpu()
)

dummy = torch.randn(
    1,
    SEQ_LEN,
    FEATURE_DIM,
    dtype=torch.float32,
)

TRACED_PATH = (
    OUTPUT_DIR
    / "wl_bisindo_hand134_transformer_traced.pt"
)

mha_fastpath_available = (
    hasattr(
        torch.backends,
        "mha"
    )
    and hasattr(
        torch.backends.mha,
        "get_fastpath_enabled"
    )
    and hasattr(
        torch.backends.mha,
        "set_fastpath_enabled"
    )
)

previous_fastpath = None

if mha_fastpath_available:
    previous_fastpath = (
        torch.backends.mha
        .get_fastpath_enabled()
    )

    torch.backends.mha.set_fastpath_enabled(
        False
    )

    print(
        "MHA fast-path temporarily disabled."
    )

try:
    with torch.inference_mode():
        traced = torch.jit.trace(
            export_model,
            dummy,
            strict=False,
            check_trace=True,
        )

    traced.save(
        str(
            TRACED_PATH
        )
    )

finally:
    if (
        mha_fastpath_available
        and previous_fastpath
        is not None
    ):
        torch.backends.mha.set_fastpath_enabled(
            previous_fastpath
        )

traced = torch.jit.load(
    str(
        TRACED_PATH
    ),
    map_location="cpu"
).eval()

parity_diffs = []

for batch_size in [
    1,
    4
]:
    sample = torch.randn(
        batch_size,
        SEQ_LEN,
        FEATURE_DIM,
        dtype=torch.float32,
    )

    with torch.inference_mode():
        ref = export_model(
            sample
        )

        out = traced(
            sample
        )

    diff = float(
        torch.max(
            torch.abs(
                ref
                - out
            )
        )
    )

    parity_diffs.append(
        diff
    )

    print(
        f"Parity batch="
        f"{batch_size}: "
        f"{diff:.8f}"
    )

max_diff = max(
    parity_diffs
)

assert max_diff < 1e-4, (
    f"TorchScript parity failed: "
    f"{max_diff}"
)

print(
    "✅ TorchScript export PASSED:",
    TRACED_PATH
)

## Save deployment config + final ZIP

In [ ]:
with open(
    OUTPUT_DIR
    / "class_mapping.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            str(k): v
            for k, v
            in LABELS_ID.items()
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

deployment_config = {
    "model": (
        "DualHandTemporalTransformer"
    ),
    "seq_len": SEQ_LEN,
    "feature_dim": FEATURE_DIM,
    "num_classes": NUM_CLASSES,
    "test_signer": TEST_SIGNER,
    "development_split_mode": (
        SPLIT_MODE
    ),
    "development_validation_signer": (
        DEV_VAL_SIGNER
    ),
    "best_epoch": int(
        best_epoch
    ),
    "test_accuracy": float(
        test_acc
    ),
    "test_macro_f1": float(
        test_f1
    ),
    "zero_f1_classes": (
        zero_f1[
            "class_name"
        ].tolist()
    ),
}

with open(
    OUTPUT_DIR
    / "deployment_config.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        deployment_config,
        f,
        ensure_ascii=False,
        indent=2,
    )

summary = pd.DataFrame({
    "Metric": [
        "Best Development Epoch",
        "Best Development Macro-F1",
        "Final Test Accuracy",
        "Final Test Macro-F1",
        "Zero-F1 Classes",
        "Test Signer",
    ],
    "Value": [
        best_epoch,
        round(
            best_val_f1,
            4
        ),
        round(
            float(
                test_acc
            ),
            4
        ),
        round(
            float(
                test_f1
            ),
            4
        ),
        len(
            zero_f1
        ),
        TEST_SIGNER,
    ],
})

summary.to_csv(
    OUTPUT_DIR
    / "final_summary.csv",
    index=False
)

display(
    summary
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "WL_BISINDO_HAND134_TRANSFORMER_V4_OUTPUTS.zip"
)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for p in OUTPUT_DIR.iterdir():
        if p.is_file():
            zf.write(
                p,
                arcname=p.name
            )

print(
    "✅ DONE:",
    ZIP_PATH
)